# Part 2 — Data Models & Prompt Templates

Questo notebook dimostra il corretto funzionamento dei moduli di modellazione dati e prompt (EP-02, EP-06, EP-11, EP-15):

| File | Responsabilità |
|------|----------------|
| `src/models/schemas.py` | Tutti i modelli Pydantic v2 del pipeline |
| `src/models/state.py` | `BuilderState` e `QueryState` (`TypedDict`) per LangGraph |
| `src/prompts/templates.py` | Catalogo centralizzato dei template di prompt (PT-01 → PT-11) |
| `src/prompts/few_shot.py` | Loaders e formatter dei few-shot examples |

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. Modelli di Ingestion — `Document` e `Chunk`

`Document` rappresenta una pagina estratta da un PDF. `Chunk` è un frammento prodotto da `RecursiveCharacterTextSplitter` con metadati di provenienza.

In [2]:
from pydantic import ValidationError
from src.models.schemas import Document, Chunk

# Document: pagina grezza da PDF
doc = Document(
    text="A Customer is any individual that has registered on the platform.",
    metadata={"source": "business_glossary.pdf", "page": "1"},
)
print("Document:")
print(f"  text (first 60): '{doc.text[:60]}'")
print(f"  metadata: {doc.metadata}")
print()

# Chunk: frammento semanticamente coerente
chunk = Chunk(
    text="A Customer is any individual that has registered on the platform.",
    chunk_index=0,
    metadata={"source": "business_glossary.pdf", "page": "1", "token_count": "14"},
)
print("Chunk:")
print(f"  chunk_index: {chunk.chunk_index}")
print(f"  token_count: {chunk.metadata['token_count']}")

# Validazione Pydantic — campo obbligatorio mancante
try:
    bad = Chunk(text="test")  # manca chunk_index
except ValidationError as e:
    print(f"\n✅ ValidationError rilevata correttamente: {e.error_count()} errore/i")

Document:
  text (first 60): 'A Customer is any individual that has registered on the plat'
  metadata: {'source': 'business_glossary.pdf', 'page': '1'}

Chunk:
  chunk_index: 0
  token_count: 14

✅ ValidationError rilevata correttamente: 1 errore/i


## 2. Modelli di Estrazione — `Triplet` e `TripletExtractionResponse`

`Triplet` cattura un fatto semantico (subject, predicate, object) con provenienza verbatim e confidence. `TripletExtractionResponse` è lo schema di output JSON dell'SLM.

In [3]:
from src.models.schemas import Triplet, TripletExtractionResponse

triplet = Triplet(
    subject="Customer",
    predicate="places",
    object="SalesOrder",
    provenance_text="A Customer places one or more SalesOrders.",
    confidence=1.0,
    source_chunk_index=0,
)
print("Triplet:")
print(f"  ({triplet.subject}, {triplet.predicate}, {triplet.object})")
print(f"  provenance: '{triplet.provenance_text}'")
print(f"  confidence: {triplet.confidence}")

# Risposta composita dell'SLM
response = TripletExtractionResponse(
    triplets=[
        triplet,
        Triplet(
            subject="SalesOrder",
            predicate="contains",
            object="OrderLineItem",
            provenance_text="A SalesOrder contains one or more OrderLineItems.",
            confidence=1.0,
        ),
    ]
)
print(f"\nTripletExtractionResponse: {len(response.triplets)} triplet")
for t in response.triplets:
    print(f"  ({t.subject}, {t.predicate}, {t.object}) [conf={t.confidence}]")

# Validazione confidence range [0.0, 1.0]
try:
    bad = Triplet(
        subject="X", predicate="y", object="Z",
        provenance_text="...", confidence=1.5  # fuori range
    )
except ValidationError as e:
    print(f"\n✅ Confidence > 1.0 rilevata: {e.errors()[0]['msg']}")

Triplet:
  (Customer, places, SalesOrder)
  provenance: 'A Customer places one or more SalesOrders.'
  confidence: 1.0

TripletExtractionResponse: 2 triplet
  (Customer, places, SalesOrder) [conf=1.0]
  (SalesOrder, contains, OrderLineItem) [conf=1.0]

✅ Confidence > 1.0 rilevata: Input should be less than or equal to 1


## 3. Entity Resolution — `EntityCluster`, `CanonicalEntityDecision`, `Entity`

In [4]:
from src.models.schemas import EntityCluster, CanonicalEntityDecision, Entity

# Cluster di kandidati duplicati trovato dal blocking K-NN
cluster = EntityCluster(
    canonical_candidate="Customer",
    variants=["Customer", "customer", "Client", "client"],
    avg_similarity=0.91,
)
print("EntityCluster:")
print(f"  canonical_candidate: {cluster.canonical_candidate}")
print(f"  variants: {cluster.variants}")
print(f"  avg_similarity: {cluster.avg_similarity}")

# Decisione del LLM Judge
decision = CanonicalEntityDecision(
    merge=True,
    canonical_name="Customer",
    reasoning="Tutte le varianti riferiscono allo stesso concetto di entità business: il cliente della piattaforma.",
)
print(f"\nCanonicalEntityDecision: merge={decision.merge}, canonical='{decision.canonical_name}'")

# Entità risolta pronta per RAG Mapping
entity = Entity(
    name="Customer",
    definition="Any individual or legal entity that has registered and made at least one purchase.",
    synonyms=["client", "buyer", "account holder"],
    provenance_text="A Customer is any individual or legal entity that has registered an account.",
    source_doc="business_glossary.pdf",
)
print(f"\nEntity: '{entity.name}'")
print(f"  definition: {entity.definition[:60]}...")
print(f"  synonyms: {entity.synonyms}")
print(f"  embedding: {'<vuoto — non ancora calcolato>' if entity.embedding is None else len(entity.embedding)}")

EntityCluster:
  canonical_candidate: Customer
  variants: ['Customer', 'customer', 'Client', 'client']
  avg_similarity: 0.91

CanonicalEntityDecision: merge=True, canonical='Customer'

Entity: 'Customer'
  definition: Any individual or legal entity that has registered and made ...
  synonyms: ['client', 'buyer', 'account holder']
  embedding: <vuoto — non ancora calcolato>


## 4. Schema DDL — `ColumnSchema`, `TableSchema`, `EnrichedTableSchema`

In [5]:
from src.models.schemas import ColumnSchema, TableSchema, EnrichedTableSchema, EnrichedColumn

# Colonne di una tabella DDL
columns = [
    ColumnSchema(name="CUST_ID", data_type="INT", is_primary_key=True),
    ColumnSchema(name="CUST_NM", data_type="VARCHAR(100)"),
    ColumnSchema(name="CUST_EMAIL", data_type="VARCHAR(255)"),
    ColumnSchema(name="REG_DT", data_type="DATE"),
]

table = TableSchema(
    table_name="TB_CUST",
    schema_name="dbo",
    columns=columns,
    ddl_source="CREATE TABLE dbo.TB_CUST (CUST_ID INT PRIMARY KEY, ...);",
)
print("TableSchema:")
print(f"  {table.schema_name}.{table.table_name} — {len(table.columns)} colonne")
for col in table.columns:
    pk = " [PK]" if col.is_primary_key else ""
    print(f"  · {col.name}: {col.data_type}{pk}")

# Promozione a EnrichedTableSchema (aggiunge nomi human-readable dall'LLM)
enriched = EnrichedTableSchema.from_table_schema(table)
enriched.enriched_table_name = "Customer Table"
enriched.table_description = "Stores master data for registered customers."
enriched.enriched_columns = [
    EnrichedColumn(original_name="CUST_ID", enriched_name="Customer ID"),
    EnrichedColumn(original_name="CUST_NM", enriched_name="Customer Name"),
    EnrichedColumn(original_name="CUST_EMAIL", enriched_name="Customer Email"),
    EnrichedColumn(original_name="REG_DT", enriched_name="Registration Date"),
]
print(f"\nEnrichedTableSchema:")
print(f"  enriched_table_name: '{enriched.enriched_table_name}'")
print(f"  description: '{enriched.table_description}'")
for ec in enriched.enriched_columns:
    print(f"  · {ec.original_name} → '{ec.enriched_name}'")

TableSchema:
  dbo.TB_CUST — 4 colonne
  · CUST_ID: INT [PK]
  · CUST_NM: VARCHAR(100)
  · CUST_EMAIL: VARCHAR(255)
  · REG_DT: DATE

EnrichedTableSchema:
  enriched_table_name: 'Customer Table'
  description: 'Stores master data for registered customers.'
  · CUST_ID → 'Customer ID'
  · CUST_NM → 'Customer Name'
  · CUST_EMAIL → 'Customer Email'
  · REG_DT → 'Registration Date'


## 5. Mapping & Cypher — `MappingProposal`, `CriticDecision`, `CypherStatement`

In [6]:
from src.models.schemas import MappingProposal, CriticDecision, CypherStatement, GraderDecision, RetrievedChunk

# Proposta di mapping LLM (RAG Mapper)
proposal = MappingProposal(
    table_name="TB_CUST",
    mapped_concept="Customer",
    confidence=0.95,
    reasoning="TB_CUST stores customer identity data matching the 'Customer' business concept.",
    alternative_concepts=["CustomerAddress"],
)
print("MappingProposal:")
print(f"  {proposal.table_name} → '{proposal.mapped_concept}' [conf={proposal.confidence}]")
print(f"  reasoning: {proposal.reasoning[:70]}...")

# Decisione del LLM Critic
critic = CriticDecision(
    approved=True,
    critique=None,
)
print(f"\nCriticDecision: approved={critic.approved}")

# Statement Cypher validato
cypher_stmt = CypherStatement(
    cypher="MERGE (t:PhysicalTable {name: $table_name}) MERGE (c:BusinessConcept {name: $concept_name}) MERGE (t)-[:IMPLEMENTS]->(c)",
    params={"table_name": "TB_CUST", "concept_name": "Customer"},
    mapping_id="TB_CUST__Customer",
)
print(f"\nCypherStatement [id={cypher_stmt.mapping_id}]:")
print(f"  {cypher_stmt.cypher[:80]}...")
print(f"  params: {cypher_stmt.params}")

# GraderDecision (Self-RAG)
grade = GraderDecision(grounded=True, critique=None, action="pass")
print(f"\nGraderDecision: grounded={grade.grounded}, action='{grade.action}'")

MappingProposal:
  TB_CUST → 'Customer' [conf=0.95]
  reasoning: TB_CUST stores customer identity data matching the 'Customer' business...

CriticDecision: approved=True

CypherStatement [id=TB_CUST__Customer]:
  MERGE (t:PhysicalTable {name: $table_name}) MERGE (c:BusinessConcept {name: $con...
  params: {'table_name': 'TB_CUST', 'concept_name': 'Customer'}

GraderDecision: grounded=True, action='pass'


## 6. LangGraph States — `BuilderState` e `QueryState`

I `TypedDict` sono lo schema di stato per i due grafi LangGraph. Ogni nodo riceve lo stato completo e ritorna solo i campi che aggiorna.

In [7]:
from src.models.state import BuilderState, QueryState
from src.models.schemas import Chunk, Triplet, Entity, MappingProposal
import typing

# Recupera i campi definiti in BuilderState
builder_fields = list(BuilderState.__annotations__.keys())
query_fields = list(QueryState.__annotations__.keys())

print(f"BuilderState — {len(builder_fields)} campi:")
for f in builder_fields:
    print(f"  · {f}: {BuilderState.__annotations__[f]}")

print(f"\nQueryState — {len(query_fields)} campi:")
for f in query_fields:
    print(f"  · {f}: {QueryState.__annotations__[f]}")

# Simulazione: un nodo riceve lo stato e aggiorna solo i propri campi
def mock_extract_node(state: BuilderState) -> dict:
    """Simula il nodo Extract_Triplets — aggiorna solo 'triplets'."""
    chunks = state.get("chunks", [])
    # ... logica estrazione ...
    return {"triplets": []}  # ritorna solo i campi aggiornati

initial_state: BuilderState = {
    "source_doc": "business_glossary.pdf",
    "chunks": [],
    "reflection_attempts": 0,
    "healing_attempts": 0,
    "hitl_flag": False,
    "failed_mappings": [],
    "ingestion_errors": [],
}
update = mock_extract_node(initial_state)
print(f"\nNodo ritorna solo: {list(update.keys())} (LangGraph merge nel state globale)")
print("✅ BuilderState e QueryState validati")

BuilderState — 20 campi:
  · ddl_paths: ForwardRef('list[str]', module='src.models.state')
  · source_doc: ForwardRef('str', module='src.models.state')
  · documents: ForwardRef('list[Document]', module='src.models.state')
  · chunks: ForwardRef('list[Chunk]', module='src.models.state')
  · triplets: ForwardRef('list[Triplet]', module='src.models.state')
  · entities: ForwardRef('list[Entity]', module='src.models.state')
  · tables: ForwardRef('list[TableSchema]', module='src.models.state')
  · enriched_tables: ForwardRef('list[EnrichedTableSchema]', module='src.models.state')
  · pending_tables: ForwardRef('list[EnrichedTableSchema]', module='src.models.state')
  · current_table: ForwardRef('EnrichedTableSchema | None', module='src.models.state')
  · current_entities: ForwardRef('list[Entity]', module='src.models.state')
  · mapping_proposal: ForwardRef('MappingProposal | None', module='src.models.state')
  · reflection_prompt: ForwardRef('str | None', module='src.models.state')
  · r

## 7. Prompt Templates — Catalogo PT-01 → PT-11

Tutti i prompt sono costanti di modulo in `src/prompts/templates.py`. I nodi li importano direttamente — nessun prompt inline nelle funzioni.

In [8]:
from src.prompts import templates

# Lista di tutti i template disponibili
template_constants = [name for name in dir(templates) if name.isupper() and not name.startswith("_")]
print(f"Template disponibili ({len(template_constants)}):")
for name in template_constants:
    val = getattr(templates, name)
    preview = val.replace("\n", " ")[:80]
    print(f"  {name}: '{preview}...'" if len(val) > 80 else f"  {name}: '{val}'")

print()
# Mostra EXTRACTION_SYSTEM completo
print("=" * 60)
print("PT-01: EXTRACTION_SYSTEM (prime 10 righe)")
print("=" * 60)
for line in templates.EXTRACTION_SYSTEM.split("\n")[:10]:
    print(f"  {line}")

Template disponibili (20):
  ANSWER_SYSTEM: 'You are a precise data governance analyst assistant. You answer questions about ...'
  ANSWER_USER: 'Answer the following question using only the retrieved context below.  <retrieve...'
  ANSWER_WITH_CRITIQUE_USER: 'Your previous answer contained unsupported claims. A factual audit identified th...'
  CRITIC_SYSTEM: 'You are an adversarial data governance auditor. Your role is to find faults in p...'
  CRITIC_USER: 'Audit the following mapping proposal.  <mapping_proposal> {proposal_json} </mapp...'
  CYPHER_FIX_USER: 'The following Cypher statement failed execution with a Neo4j error.  <broken_cyp...'
  CYPHER_SYSTEM: 'You are a Neo4j Cypher expert specialised in knowledge graph construction for da...'
  CYPHER_USER: 'Generate the MERGE Cypher statements for the following mapping.  <few_shot_examp...'
  ENRICHMENT_SYSTEM: 'You are a database naming expert specialised in corporate and enterprise data sc...'
  ENRICHMENT_USER: 'Expand the abb

In [9]:
# Mostra come EXTRACTION_USER viene formattato con il testo reale
sample_text = "A Customer places one or more SalesOrders. Each SalesOrder contains one or more OrderLineItems."

formatted_extraction_user = templates.EXTRACTION_USER.format(chunk_text=sample_text)
print("PT-01: EXTRACTION_USER (formattato):")
print("-" * 60)
print(formatted_extraction_user)
print("-" * 60)

# Verifica che il placeholder sia stato sostituito
assert "{chunk_text}" not in formatted_extraction_user
assert sample_text in formatted_extraction_user
print("\n✅ Placeholder sostituito correttamente")

PT-01: EXTRACTION_USER (formattato):
------------------------------------------------------------
Extract all factual triplets from the following text passage.

<text_passage>
A Customer places one or more SalesOrders. Each SalesOrder contains one or more OrderLineItems.
</text_passage>

Remember: copy provenance_text verbatim. Return only the JSON object.
------------------------------------------------------------

✅ Placeholder sostituito correttamente


## 8. Few-Shot Examples — `load_cypher_examples()` e `load_mapping_examples()`

In [10]:
from src.prompts.few_shot import (
    load_cypher_examples,
    load_mapping_examples,
    format_cypher_examples,
    format_mapping_examples,
)

# Carica esempi Cypher
cypher_examples = load_cypher_examples(n=3)
print(f"Cypher examples caricati: {len(cypher_examples)}")
for ex in cypher_examples:
    print(f"  · [{ex.concept_name}] {ex.description[:60]}")

# Carica esempi Mapping
mapping_examples = load_mapping_examples(n=2)
print(f"\nMapping examples caricati: {len(mapping_examples)}")
for ex in mapping_examples:
    print(f"  · [{ex.concept_name}] DDL snippet: {ex.ddl_snippet[:50]}...")

Cypher examples caricati: 3
  · [Customer] Customer master table mapping
  · [Product] Product catalogue table mapping
  · [SalesOrder] Sales order header mapping with STATUS constraint

Mapping examples caricati: 2
  · [Customer] DDL snippet: CUSTOMER_MASTER: CUST_ID (INT, PK), FULL_NAME (VAR...
  · [Product] DDL snippet: TB_PRODUCT: PRODUCT_ID (INT, PK), SKU (VARCHAR, UN...


In [11]:
# Formatta i few-shot examples come testo per il prompt
cypher_block = format_cypher_examples(cypher_examples[:2])
print("Few-shot Cypher (formattato per CYPHER_USER):")
print("-" * 60)
print(cypher_block[:600] + "..." if len(cypher_block) > 600 else cypher_block)
print("-" * 60)

mapping_block = format_mapping_examples(mapping_examples[:1])
print("\nFew-shot Mapping (formattato per MAPPING_USER):")
print("-" * 60)
print(mapping_block[:400] + "..." if len(mapping_block) > 400 else mapping_block)
print("-" * 60)

print("\n✅ Few-shot examples caricati e formattati correttamente")

Few-shot Cypher (formattato per CYPHER_USER):
------------------------------------------------------------
Example 1: Customer master table mapping
DDL: CREATE TABLE CUSTOMER_MASTER (CUST_ID INT PRIMARY KEY, FULL_NAME VARCHAR(200), EMAIL VARCHAR(150), REGION_CODE VARCHAR(10))
Concept: Customer
Cypher:
MERGE (bc:BusinessConcept {name: $concept_name})
ON CREATE SET bc.definition = $definition, bc.provenance_text = $provenance_text, bc.source_doc = $source_doc, bc.synonyms = $synonyms, bc.confidence_score = $confidence_score
ON MATCH SET bc.confidence_score = $confidence_score

MERGE (pt:PhysicalTable {table_name: $table_name})
ON CREATE SET pt.schema_name = $schema_name, pt.column_names = $column_nam...
------------------------------------------------------------

Few-shot Mapping (formattato per MAPPING_USER):
------------------------------------------------------------
Example 1:
Table DDL: CUSTOMER_MASTER: CUST_ID (INT, PK), FULL_NAME (VARCHAR), EMAIL (VARCHAR), REGION_CODE (VARCHAR),

## Riepilogo — Architettura di Part 2

```
┌─────────────────────────────────────────────────────────────────────┐
│                   EP-02/06/11/15 — Data Models & Prompts            │
├──────────────────────┬──────────────────────┬───────────────────────┤
│    schemas.py        │     state.py         │  templates.py         │
│                      │                      │  + few_shot.py        │
│  Document, Chunk     │  BuilderState        │  PT-01 Extraction     │
│  Triplet             │  QueryState          │  PT-02 ER Judge       │
│  Entity, Cluster     │  (TypedDict)         │  PT-03 Mapping        │
│  TableSchema         │                      │  PT-04/05 Actor-Critic│
│  EnrichedTable       │  total=False →       │  PT-06/07 Cypher      │
│  MappingProposal     │  campi opzionali     │  PT-08/09 Answer      │
│  CriticDecision      │  per LangGraph       │  PT-10 Grader         │
│  CypherStatement     │  reducers            │  PT-11 Enrichment     │
│  GraderDecision      │                      │                       │
│  EvaluationReport    │                      │  few_shot_cypher.json │
│                      │                      │  few_shot_mapping.json│
└──────────────────────┴──────────────────────┴───────────────────────┘
```

**Principi architetturali:**
- **Un solo file** per tutti i modelli Pydantic → import sempre inequivocabili
- **TypedDict con `total=False`** → tutti i campi opzionali (flessibilità dei nodi LangGraph)
- **Prompt come costanti di modulo** → nessun prompt inline nelle funzioni
- **JSON seed files** per i few-shot → versionabili con Git, modificabili senza toccare il codice